FIRST ADJUSTMENTS

In [31]:
import pandas as pd
# ===============================
# 2. LOAD DATASETS 
# ===============================
glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')
activity = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Activity Data/UoMActivity2301.csv')



glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)
activity.rename(columns={'activity_ts': 'timestamp'}, inplace=True)


glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')
activity["timestamp"] = pd.to_datetime(activity["timestamp"], dayfirst=True, errors='coerce')


CASE ID: MEAL

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta


base_dir = "caseid_meal"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)



import pandas as pd

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')
activity = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Activity Data/UoMActivity2301.csv')
sleeptime = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Sleep Data/UoM2301sleeptime.csv')




# Rename time columns 
glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)
activity.rename(columns={'activity_ts': 'timestamp'}, inplace=True)
sleeptime.rename(columns={'start_date_ts': 'timestamp'}, inplace=True)

# Convert to datetime (format DD/MM/YYYY)
glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')
activity["timestamp"] = pd.to_datetime(activity["timestamp"], dayfirst=True, errors='coerce')
sleeptime["timestamp"] = pd.to_datetime(sleeptime["timestamp"], dayfirst=True, errors='coerce')


# case day is when the person went to bed
sleeptime["date"] = sleeptime["timestamp"].dt.date
sleeptime = sleeptime.dropna(subset=["timestamp", "duration_in_sec"])


# SEDENTARY activities are removed
activity = activity[activity["activity_type"] != "SEDENTARY"].reset_index(drop=True)


for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus), ("activity_type", activity), ("start_date_ts", sleeptime)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"{invalid} timestamps inválidos encontrados em {name} (convertidos em NaT)")


# ===============================
# 2.5 CREATE UNIQUE IDS
# ===============================

glucose["id"] = range(1, len(glucose) + 1)
meals["id"]   = range(1, len(meals) + 1)
bolus["id"]   = range(1, len(bolus) + 1)
activity["id"]= range(1, len(activity) + 1)


# ===============================
# 3. AUXILIARY FUNCTIONS
# ===============================

def get_next_meal(current_meal, all_meals):
    """Devolve a refeição seguinte."""
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    

    # Filter data from meal
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]

    # Search possible return to baseline
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        # if return to early, ignore
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    # period post-prandial until 3h (or until valid return)
    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None

    # Search for absolute max in that period
    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time



def find_recovery(glucose_df, peak_time, baseline, end_time):
    
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) & 
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]



# ====================================
# TRANSFORM ACTIVITY INTO INTERVALS
# ====================================

activity_sorted = activity.sort_values("timestamp").reset_index(drop=True)

activity_intervals = []
current_type = None
current_start = None

for i, row in activity_sorted.iterrows():
    if current_type is None:
        # start of first activity
        current_type = row["activity_type"]
        current_start = row["timestamp"]
        last_time = row["timestamp"]
        continue

    # if activity change → close previous interval
    if row["activity_type"] != current_type:
        activity_intervals.append({
            "activity_type": current_type,
            "start": current_start,
            "end": last_time
        })
        # iniciate new interval
        current_type = row["activity_type"]
        current_start = row["timestamp"]

    last_time = row["timestamp"]

# close last intervals
activity_intervals.append({
    "activity_type": current_type,
    "start": current_start,
    "end": last_time
})

activity_intervals = pd.DataFrame(activity_intervals)


# ===============================
# 4. CREATE COMPORTAMENTAL EVENTS
# ===============================
events = []

# sleep events

for _, s in sleeptime.iterrows():

    case_id = str(s["date"])   # day when person went to sleep

    events.append({
        "case_id": case_id,
        "event_type": "SleepTime",
        "timestamp": s["timestamp"],           # time to go to sleep
        "value": s["duration_in_sec"]           # sleep total 
    })



for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)
    
    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]
    
    glucose_window = glucose[(glucose["timestamp"] >= start_time) & (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    # Baseline = average glucose in the 1h window before each meal
    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    # Event: Meal
    events.append({
        "case_id": meal["id"],
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Event: Bolus (if appliable)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": meal["id"],
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })




    # ===============================
    # EVENTS START AND END ACTIVITY
    # ===============================

    acts = activity_intervals[
        (activity_intervals["end"] >= start_time - timedelta(minutes=30)) &
        (activity_intervals["start"] <= end_time)
    ]

    for _, a in acts.iterrows():
        # Activity start
        events.append({
            "case_id": meal["id"],
            "event_type": f"Activity",
            "timestamp": max(a["start"], start_time - timedelta(minutes=30)),
            "value": None
        })


    
    # Event: Peak of glicose
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Event: Return to normal
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })

    # ===============================
    # 5. GENERATE GRAPH
    # ===============================
    plt.figure(figsize=(10,5))
    
    # Plot glicose
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose", zorder=1)
    
    # Meal
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)


    # ===============================
    # Activity markers (start/end)
    # ===============================
    activity_for_graph = activity_intervals[
        (activity_intervals["end"] >= start_time - timedelta(minutes=30)) &
        (activity_intervals["start"] <= end_time)
    ]

    for _, a in activity_for_graph.iterrows():

        # start of activiy → purple
        plt.scatter(
            a["start"],
            baseline - 0.5,        
            color="purple",
            s=80,
            marker="o",
            zorder=4,
            label="Activity Start"
        )
        
        # end of activity → black
        plt.scatter(
            a["end"],
            baseline - 0.5,
            color="black",
            s=80,
            marker="o",
            zorder=4,
            label="Activity End"
        )

    
    # Peak (triangle + peak value)
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    # Return to baseline
    if recovery_time and (peak_time is None or recovery_time > peak_time):
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)

    # Baseline
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    # Bolus insulin (blue)
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline, color='blue', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.2, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)

    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial glucose response - meal {meal['id']} at {meal['timestamp'].strftime('%d/%m/%Y %H:%M') if pd.notna(meal['timestamp']) else 'unknown time'}")
    
    # remove duplicate labels
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal/processed/meal_{meal['id']}_curve.png")
    plt.close() 


# ===============================
# 6. SAVE EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_meal/processed/event_log.csv", index=False)

print("Concluded!")
print(f"Created events: {len(events_df)}")
print("Saved at: caseid_meal/processed/event_log.csv")


CASE ID: DAY    

In [32]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta


base_dir = "caseid_day"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)



glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')
activity = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Activity Data/UoMActivity2301.csv')
sleeptime = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Sleep Data/UoM2301sleeptime.csv')


glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)
activity.rename(columns={'activity_ts': 'timestamp'}, inplace=True)
sleeptime.rename(columns={'start_date_ts': 'timestamp'}, inplace=True)


glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')
activity["timestamp"] = pd.to_datetime(activity["timestamp"], dayfirst=True, errors='coerce')
sleeptime['timestamp'] = pd.to_datetime(sleeptime['timestamp'], dayfirst=True, errors='coerce')


activity = activity[activity["activity_type"] != "SEDENTARY"].reset_index(drop=True)


sleeptime["date"] = sleeptime["timestamp"].dt.date
sleeptime = sleeptime.dropna(subset=["timestamp", "duration_in_sec"])


glucose["date"] = glucose["timestamp"].dt.date
meals["date"]   = meals["timestamp"].dt.date
bolus["date"]   = bolus["timestamp"].dt.date
activity["date"]= activity["timestamp"].dt.date


for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus), ("activity_type", activity), ("start_date_ts", sleeptime)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f" {invalid} invalid timestamps found at {name} (converted into NaT)")



def get_next_meal(current_meal, all_meals):
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None, None

    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time

def find_recovery(glucose_df, peak_time, baseline, end_time):
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) &
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]



activity_sorted = activity.sort_values("timestamp").reset_index(drop=True)

activity_intervals = []
current_type = None
current_start = None

for i, row in activity_sorted.iterrows():
    if current_type is None:
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]
        last_time = row["timestamp"]
        continue

    
    if row["activity_type"] != current_type:
        activity_intervals.append({
            "activity_type": current_type,
            "start": current_start,
            "end": last_time
        })
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]

    last_time = row["timestamp"]


activity_intervals.append({
    "activity_type": current_type,
    "start": current_start,
    "end": last_time
})

activity_intervals = pd.DataFrame(activity_intervals)


events = []



for _, s in sleeptime.iterrows():

    case_id = str(s["date"])  

    events.append({
        "case_id": case_id,
        "event_type": "SleepTime",
        "timestamp": s["timestamp"],           
        "value": s["duration_in_sec"]          
    })


for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)

    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]

    glucose_window = glucose[(glucose["timestamp"] >= start_time) &
                             (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    case_id = str(meal["date"])  

    # Meal event
    events.append({
        "case_id": case_id,
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Bolus event
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & 
                           (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": case_id,
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })



    acts = activity_intervals[
        (activity_intervals["end"] >= start_time - timedelta(minutes=30)) &
        (activity_intervals["start"] <= end_time)
    ]

    for _, a in acts.iterrows():
        
        events.append({
            "case_id": case_id,
            "event_type": f"Activity",
            "timestamp": max(a["start"], start_time - timedelta(minutes=30)),
            "value": None
        })
        


    # Peak
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Recovery
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })


all_days = sorted(glucose["date"].unique())

for day in all_days:

    
    day_glucose = glucose[glucose["date"] == day]

    if day_glucose.empty:
        continue

    
    day_meals = meals[meals["date"] == day]

    
    day_bolus = bolus[bolus["date"] == day]

    
    plt.figure(figsize=(14, 6))

    # — Glucose curve —
    plt.plot(day_glucose["timestamp"], day_glucose["value"], '-o', label="Glucose", zorder=1)

    # — Meals —
    for _, meal in day_meals.iterrows():
        plt.axvline(meal["timestamp"], color='green', linestyle='--', alpha=0.7, label='Meal')
    
    # — Bolus insulin —
    for _, b in day_bolus.iterrows():
        plt.scatter(b["timestamp"], day_glucose["value"].min() - 0.2,
                    color='blue', marker='o', s=80, label="Bolus")
        plt.text(b["timestamp"],
                 day_glucose["value"].min() - 0.5,
                 f"{b.get('bolus_dose',0):.1f}",
                 color="blue", ha="center")

    # — PPGR peaks —
    peaks_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_peak"]
    for pk in peaks_today:
        plt.scatter(pk["timestamp"], pk["value"], color='red', marker='^', s=120, label="PPGR_peak")
        plt.text(pk["timestamp"], pk["value"] + 0.2, f"{pk['value']:.1f}", color='red', ha='center')

    # — Recovery to baseline —
    recov_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_recovery"]
    for rc in recov_today:
        plt.axvline(rc["timestamp"], color='orange', linestyle='--', label="Recovery")

    plt.title(f"Daily Glucose Profile — {day}")
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")

    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())

    plt.tight_layout()
    plt.savefig(f"caseid_day/processed/day_{day}_curve.png")
    plt.close()



events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_day/processed/event_log.csv", index=False)

print("Concluded!")
print(f"Created events: {len(events_df)}")
print("Saved at: caseid_day/processed/event_log.csv")


✅ Processamento concluído!
Eventos criados: 2524
Ficheiro guardado em: caseid_day/processed/event_log.csv


CASE ID: MEAL WITH HYPERGLYCEMIA AND HYPOGLICEMIA FILTER

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta


base_dir = "caseid_meal_HypoHyper"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)



import pandas as pd

glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')
activity = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Activity Data/UoMActivity2301.csv')


glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)
activity.rename(columns={'activity_ts': 'timestamp'}, inplace=True)


glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')
activity["timestamp"] = pd.to_datetime(activity["timestamp"], dayfirst=True, errors='coerce')

activity = activity[activity["activity_type"] != "SEDENTARY"].reset_index(drop=True)


for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus), ("activity_type", activity)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f"{invalid} invalid timestamps found at {name} (converted into NaT)")




glucose["id"] = range(1, len(glucose) + 1)
meals["id"]   = range(1, len(meals) + 1)
bolus["id"]   = range(1, len(bolus) + 1)
activity["id"]   = range(1, len(activity) + 1)



def get_next_meal(current_meal, all_meals):
    
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
 

    
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]

    
    return_to_baseline = post_meal[post_meal["value"] <= baseline]


    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    
    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None

    
    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time



def find_recovery(glucose_df, peak_time, baseline, end_time):
   
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) & 
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]



activity_sorted = activity.sort_values("timestamp").reset_index(drop=True)

activity_intervals = []
current_type = None
current_start = None

for i, row in activity_sorted.iterrows():
    if current_type is None:
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]
        last_time = row["timestamp"]
        continue

    
    if row["activity_type"] != current_type:
        activity_intervals.append({
            "activity_type": current_type,
            "start": current_start,
            "end": last_time
        })
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]

    last_time = row["timestamp"]

# fechar o último intervalo
activity_intervals.append({
    "activity_type": current_type,
    "start": current_start,
    "end": last_time
})

activity_intervals = pd.DataFrame(activity_intervals)



events = []

for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)
    
    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]
    
    glucose_window = glucose[(glucose["timestamp"] >= start_time) & (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    # Baseline = average glucose in the 1h window before each meal
    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    
    events.append({
        "case_id": meal["id"],
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": meal["id"],
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })




    acts = activity_intervals[
        (activity_intervals["end"] >= start_time - timedelta(minutes=30)) &
        (activity_intervals["start"] <= end_time)
    ]

    for _, a in acts.iterrows():
        
        events.append({
            "case_id": meal["id"],
            "event_type": f"Activity",
            "timestamp": max(a["start"], start_time - timedelta(minutes=30)),
            "value": None
        })
        
      
    
    
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    
    
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": meal["id"],
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })


    glucose_case = glucose[(glucose["timestamp"] >= start_time) &
                           (glucose["timestamp"] <= end_time)].sort_values("timestamp")

    hyper_threshold = 7.8
    hypo_threshold = 3.9

    in_hyper = False
    in_hypo = False

    for _, row in glucose_case.iterrows():
        g = row["value"]
        t = row["timestamp"]

        # ---------- Hyperglycemia ----------
        if g > hyper_threshold and not in_hyper:
            # Start event
            events.append({
                "case_id": meal["id"],
                "event_type": "HyperglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hyper = True

        if g <= hyper_threshold and in_hyper:
            events.append({
                "case_id": meal["id"],
                "event_type": "HyperglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hyper = False

        # ---------- Hypoglycemia ----------
        if g < hypo_threshold and not in_hypo:
            events.append({
                "case_id": meal["id"],
                "event_type": "HypoglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hypo = True

        if g >= hypo_threshold and in_hypo:
            events.append({
                "case_id": meal["id"],
                "event_type": "HypoglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hypo = False


    
    plt.figure(figsize=(10,5))
    
    
    plt.plot(glucose_window["timestamp"], glucose_window["value"], '-o', label="Glucose", zorder=1)
    
    
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    

    if recovery_time and (peak_time is None or recovery_time > peak_time):
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)

    
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline, color='blue', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.2, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)

    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial Glucose Response - Meal {meal['id']}")
    
    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal_HypoHyper/processed/meal_{meal['id']}_curve.png")
    plt.close() 

    plt.figure(figsize=(10,5))

    
    gw = glucose_window.sort_values("timestamp")
    hyper_threshold = 7.8
    hypo_threshold  = 3.9
    
    ts = gw["timestamp"].values
    vs = gw["value"].values
    
    for i in range(len(gw) - 1):
        t0, t1 = ts[i], ts[i+1]
        v0, v1 = vs[i], vs[i+1]
    
        
        if v0 > hyper_threshold or v1 > hyper_threshold or v0 < hypo_threshold or v1 < hypo_threshold:
            color = "red"
        else:
            color = "blue"
    
        plt.plot([t0, t1], [v0, v1], color=color, linewidth=2)
    
    
    plt.axvline(start_time, color='green', linestyle='--', label='Meal', zorder=2)
    
    
    plt.scatter(peak_time, peak_value, color='red', marker='^', s=100, label='Peak', zorder=3)
    plt.text(peak_time, peak_value + 0.2, f"{peak_value:.1f}", color='red', ha='center', zorder=4)
    
    
    if recovery_time:
        plt.axvline(recovery_time, color='orange', linestyle='--', label='Recovery', zorder=2)
    
    
    plt.axhline(baseline, color='gray', linestyle=':', label=f'Baseline ({baseline:.1f})', zorder=1)
    
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=15)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=15))]
    for _, b in bolus_for_meal.iterrows():
        plt.scatter(b["timestamp"], baseline + 0.3, color='green', marker='o', s=80, label='Bolus Insulin', zorder=3)
        plt.text(b["timestamp"], baseline + 0.6, f"{b.get('bolus_dose', 0):.1f}", color='blue', ha='center', zorder=4)
    
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")
    plt.title(f"Postprandial glucose response - meal {meal['id']} at {meal['timestamp'].strftime('%d/%m/%Y %H:%M') if pd.notna(meal['timestamp']) else 'unknown time'}")

    
    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    
    plt.tight_layout()
    plt.savefig(f"caseid_meal_HypoHyper/processed/meal_{meal['id']}_curve.png")
    plt.close()



events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_meal_HypoHyper/processed/event_log.csv", index=False)

print("Concluded!")
print(f"Created events: {len(events_df)}")
print("Saved at: caseid_meal_HypoHyper/processed/event_log.csv")


CASE ID: DAY WITH HYPERGLYCEMIA AND HYPOGLICEMIA FILTER

In [33]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

base_dir = "caseid_day_HypoHyper"
dirs = ["graphs", "processed"]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)



glucose = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Glucose Data/UoMGlucose2301.csv')
meals   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Nutrition Data/UoMNutrition2301.csv')
bolus   = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Insulin Data/Bolus Data/UoMBolus2301.csv')
activity = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Activity Data/UoMActivity2301.csv')
sleeptime = pd.read_csv('minha_pasta/sharpic-ManchesterCSCoordinatedDiabetesStudy-fdbd74f/Sleep Data/UoM2301sleeptime.csv')


glucose.rename(columns={'bg_ts': 'timestamp'}, inplace=True)
meals.rename(columns={'meal_ts': 'timestamp'}, inplace=True)
bolus.rename(columns={'bolus_ts': 'timestamp'}, inplace=True)
activity.rename(columns={'activity_ts': 'timestamp'}, inplace=True)
sleeptime.rename(columns={'start_date_ts': 'timestamp'}, inplace=True)


glucose["timestamp"] = pd.to_datetime(glucose["timestamp"], dayfirst=True, errors='coerce')
meals["timestamp"] = pd.to_datetime(meals["timestamp"], dayfirst=True, errors='coerce')
bolus["timestamp"] = pd.to_datetime(bolus["timestamp"], dayfirst=True, errors='coerce')
activity["timestamp"] = pd.to_datetime(activity["timestamp"], dayfirst=True, errors='coerce')
sleeptime['timestamp'] = pd.to_datetime(sleeptime['timestamp'], dayfirst=True, errors='coerce')


activity = activity[activity["activity_type"] != "SEDENTARY"].reset_index(drop=True)


sleeptime["date"] = sleeptime["timestamp"].dt.date
sleeptime = sleeptime.dropna(subset=["timestamp", "duration_in_sec"])


glucose["date"] = glucose["timestamp"].dt.date
meals["date"]   = meals["timestamp"].dt.date
bolus["date"]   = bolus["timestamp"].dt.date
activity["date"]   = activity["timestamp"].dt.date


for name, df in [("value", glucose), ("meal_type", meals), ("bolus_dose", bolus), ("activity_type", activity), ("start_date_ts", sleeptime)]:
    invalid = df["timestamp"].isna().sum()
    if invalid > 0:
        print(f" {invalid} invalid timestamps found at {name} (converted into NaT)")



def get_next_meal(current_meal, all_meals):
    later_meals = all_meals[all_meals["timestamp"] > current_meal["timestamp"]]
    if later_meals.empty:
        return None
    return later_meals.iloc[0]

def find_peak(glucose_window, start_time, baseline, hours_post_meal=3):
    post_meal = glucose_window[glucose_window["timestamp"] >= start_time]
    return_to_baseline = post_meal[post_meal["value"] <= baseline]

    if not return_to_baseline.empty:
        end_time = return_to_baseline.iloc[0]["timestamp"]
        if (end_time - start_time) < timedelta(minutes=30):
            end_time = start_time + timedelta(hours=hours_post_meal)
    else:
        end_time = start_time + timedelta(hours=hours_post_meal)

    window = post_meal[
        (post_meal["timestamp"] >= start_time) &
        (post_meal["timestamp"] <= end_time)
    ]

    if window.empty:
        return None, None, None

    idx = window["value"].idxmax()
    return window.loc[idx, "timestamp"], window.loc[idx, "value"], end_time

def find_recovery(glucose_df, peak_time, baseline, end_time):
    if peak_time is None:
        return None
    post_peak = glucose_df[
        (glucose_df["timestamp"] > peak_time) &
        (glucose_df["timestamp"] <= end_time)
    ]
    post_peak_below = post_peak[post_peak["value"] <= baseline]
    if post_peak_below.empty:
        return None
    return post_peak_below.iloc[0]["timestamp"]



activity_sorted = activity.sort_values("timestamp").reset_index(drop=True)

activity_intervals = []
current_type = None
current_start = None

for i, row in activity_sorted.iterrows():
    if current_type is None:
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]
        last_time = row["timestamp"]
        continue

    
    if row["activity_type"] != current_type:
        activity_intervals.append({
            "activity_type": current_type,
            "start": current_start,
            "end": last_time
        })
        
        current_type = row["activity_type"]
        current_start = row["timestamp"]

    last_time = row["timestamp"]


activity_intervals.append({
    "activity_type": current_type,
    "start": current_start,
    "end": last_time
})

activity_intervals = pd.DataFrame(activity_intervals)

events = []

for _, s in sleeptime.iterrows():

    case_id = str(s["date"])   

    events.append({
        "case_id": case_id,
        "event_type": "SleepTime",
        "timestamp": s["timestamp"],           
        "value": s["duration_in_sec"]           
    })


hyper_threshold = 7.8
hypo_threshold  = 3.9


for _, meal in meals.iterrows():
    start_time = meal["timestamp"]
    end_time = start_time + timedelta(hours=3)

    next_meal = get_next_meal(meal, meals)
    if next_meal is not None and next_meal["timestamp"] < end_time:
        end_time = next_meal["timestamp"]

    glucose_window = glucose[(glucose["timestamp"] >= start_time) &
                             (glucose["timestamp"] <= end_time)]
    if glucose_window.empty:
        continue

    glucose_before = glucose[(glucose["timestamp"] >= start_time - timedelta(minutes=60)) &
                             (glucose["timestamp"] < start_time)]
    baseline = glucose_before["value"].mean() if not glucose_before.empty else glucose_window.iloc[0]["value"]

    case_id = str(meal["date"])   

    # Meal event
    events.append({
        "case_id": case_id,
        "event_type": "Meal",
        "timestamp": start_time,
        "value": None
    })

    # Bolus event
    bolus_for_meal = bolus[(bolus["timestamp"] >= start_time - timedelta(minutes=30)) &
                           (bolus["timestamp"] <= start_time + timedelta(minutes=180)) & 
                           (bolus["bolus_dose"] > 0)]
    for _, b in bolus_for_meal.iterrows():
        events.append({
            "case_id": case_id,
            "event_type": "BolusInsulin",
            "timestamp": b["timestamp"],
            "value": b.get("bolus_dose", None)
        })



    acts = activity_intervals[
        (activity_intervals["end"] >= start_time - timedelta(minutes=30)) &
        (activity_intervals["start"] <= end_time)
    ]

    for _, a in acts.iterrows():
        # Início da atividade
        events.append({
            "case_id": case_id,
            "event_type": f"Activity",
            "timestamp": max(a["start"], start_time - timedelta(minutes=30)),
            "value": None
        })
        
     

    # Peak
    peak_time, peak_value, computed_end_time = find_peak(glucose_window, start_time, baseline)
    if peak_time is not None:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_peak",
            "timestamp": peak_time,
            "value": peak_value
        })

    # Recovery
    recovery_time = find_recovery(glucose, peak_time, baseline, computed_end_time)
    if recovery_time:
        events.append({
            "case_id": case_id,
            "event_type": "PPGR_recovery",
            "timestamp": recovery_time,
            "value": baseline
        })




for day, day_glucose in glucose.groupby("date"):

    day_glucose = day_glucose.sort_values("timestamp")
    case_id = str(day)

    in_hyper = False
    in_hypo  = False

    for _, row in day_glucose.iterrows():
        g = row["value"]
        t = row["timestamp"]

        # ---------- Hyperglycemia ----------
        if g > hyper_threshold and not in_hyper:
            events.append({
                "case_id": case_id,
                "event_type": "HyperglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hyper = True

        if g <= hyper_threshold and in_hyper:
            events.append({
                "case_id": case_id,
                "event_type": "HyperglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hyper = False

        # ---------- Hypoglycemia ----------
        if g < hypo_threshold and not in_hypo:
            events.append({
                "case_id": case_id,
                "event_type": "HypoglycemiaStart",
                "timestamp": t,
                "value": g
            })
            in_hypo = True
        
        if g >= hypo_threshold and in_hypo:
            events.append({
                "case_id": case_id,
                "event_type": "HypoglycemiaEnd",
                "timestamp": t,
                "value": g
            })
            in_hypo = False



all_days = sorted(glucose["date"].unique())

for day in all_days:

    
    day_glucose = glucose[glucose["date"] == day]

    if day_glucose.empty:
        continue

    
    day_meals = meals[meals["date"] == day]

    
    day_bolus = bolus[bolus["date"] == day]

    
    plt.figure(figsize=(14, 6))

    # — Glucose curve —
    plt.plot(day_glucose["timestamp"], day_glucose["value"], '-o', label="Glucose", zorder=1)

    # — Meals —
    for _, meal in day_meals.iterrows():
        plt.axvline(meal["timestamp"], color='green', linestyle='--', alpha=0.7, label='Meal')
    
    # — Bolus insulin —
    for _, b in day_bolus.iterrows():
        plt.scatter(b["timestamp"], day_glucose["value"].min() - 0.2,
                    color='blue', marker='o', s=80, label="Bolus")
        plt.text(b["timestamp"],
                 day_glucose["value"].min() - 0.5,
                 f"{b.get('bolus_dose',0):.1f}",
                 color="blue", ha="center")

    # — PPGR peaks —
    peaks_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_peak"]
    for pk in peaks_today:
        plt.scatter(pk["timestamp"], pk["value"], color='red', marker='^', s=120, label="PPGR_peak")
        plt.text(pk["timestamp"], pk["value"] + 0.2, f"{pk['value']:.1f}", color='red', ha='center')

    # — Recovery to baseline —
    recov_today = [ev for ev in events if ev["case_id"] == str(day) and ev["event_type"] == "PPGR_recovery"]
    for rc in recov_today:
        plt.axvline(rc["timestamp"], color='orange', linestyle='--', label="Recovery")

    plt.title(f"Daily Glucose Profile — {day}")
    plt.xlabel("Time")
    plt.ylabel("Glucose (mmol/L)")

    
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())

    plt.tight_layout()
    plt.savefig(f"caseid_day_HypoHyper/processed/day_{day}_curve.png")
    plt.close()


# ===============================
# 6. EXPORT EVENT LOG
# ===============================
events_df = pd.DataFrame(events).sort_values(["case_id", "timestamp"])
events_df.to_csv("caseid_day_HypoHyper/processed/event_log.csv", index=False)

print("Concluded!")
print(f"Created events: {len(events_df)}")
print("Save in: caseid_day_HypoHyper/processed/event_log.csv")


✅ Processamento concluído!
Eventos criados: 3718
Ficheiro guardado em: caseid_day_HypoHyper/processed/event_log.csv
